# Incident 1 — The Infinite Quality Loop

**What happened:** A quality-checking agent evaluated content in a loop until its score exceeded 0.95. The score oscillated between 0.82 and 0.91 and never reached the threshold. The agent ran **847 iterations overnight** before someone noticed. Cost: **$16.94**. No alert fired.

**Why it happens:** The loop exit condition depends on the LLM's output hitting a threshold. Nothing bounds the number of iterations — no ceiling, no timeout, no budget.

**The fix:** A $5.00 task budget. At $0.02/call that is 250 iterations maximum. When the budget is exhausted FiGuard returns `BUDGET_EXHAUSTED` and the loop stops cleanly with a structured reason — no exception, no crash, full audit trail.

In [ ]:
import sys
!pip install "figuard>=0.3.2" --upgrade -q
for _mod in list(sys.modules.keys()):
    if _mod.startswith("figuard"):
        del sys.modules[_mod]
import figuard
print(f"✓ FiGuard {figuard.__version__} ready")

## Without FiGuard — the loop runs until you kill it

The score never reaches 0.95. The demo stops at 20 iterations. In production it ran 847.

In [ ]:
# WITHOUT FiGuard — unconstrained loop
# (demo stops at 20; in production this ran 847 times at $0.02/call)

iteration = 0
total_cost = 0.0
DEMO_LIMIT = 20

print("Starting quality check loop (no budget ceiling)...")
print()

while True:
    iteration += 1
    cost = 0.02
    total_cost += cost

    # Simulated LLM call — score oscillates, never reaches 0.95
    score = 0.85 + (iteration % 7) * 0.01

    print(f"Iteration {iteration:4d}: score={score:.2f}  cost=${cost:.2f}  total=${total_cost:.2f}")

    if score > 0.95:
        print(f"\nThreshold reached at iteration {iteration}.")
        break

    if iteration >= DEMO_LIMIT:
        print()
        print(f"--- DEMO STOPPED AT {iteration} ITERATIONS ---")
        print(f"Score never reached 0.95. In production: 847 iterations, ${847 * 0.02:.2f} spent.")
        break

## With FiGuard — hard ceiling, clean stop

A $5.00 budget on the task caps the loop at 250 iterations. FiGuard returns `BUDGET_EXHAUSTED` with a structured denial — the loop stops without crashing, and every iteration is recorded in the ledger.

In [ ]:
from figuard import FiGuardClient

client = FiGuardClient(
    api_key="sb_live_demo",
    base_url="https://figuard-sandbox-1.onrender.com",
)

cost_per_call = 0.02

budget = client.create_budget(
    user_id="quality_checker",
    total_limit=5.00,
    currency="USD",
    expires_in="1h",
    intent_context="content quality check — single document",
)

print(f"Budget: ${budget.total_limit:.2f}  "
      f"(max {int(budget.total_limit / cost_per_call)} iterations at ${cost_per_call}/call)")
print()

iteration = 0
total_cost = 0.0

while True:
    iteration += 1

    auth = client.authorize(
        session_token=budget.session_token,
        agent_id="quality_checker",
        action_type="LLM_CALL",
        description=f"Quality evaluation iteration {iteration}",
        requested_quantity=cost_per_call,
        idempotency_key=f"quality-iter-{iteration}",
    )

    if not auth.is_authorized:
        print(f"\n✓ FiGuard stopped the loop at iteration {iteration}")
        print(f"  Denial reason: {auth.denial_reason}")
        print(f"  Total spent:   ${total_cost:.2f} of ${budget.total_limit:.2f}")
        print(f"  Saved vs 847 iterations: ${(847 - iteration) * cost_per_call:.2f}")
        break

    client.confirm_event(auth.event_id, confirmed_quantity=cost_per_call)
    total_cost += cost_per_call
    score = 0.85 + (iteration % 7) * 0.01

    if iteration % 50 == 0 or iteration <= 3:
        print(f"Iteration {iteration:4d}: score={score:.2f}  spent=${total_cost:.2f} of ${budget.total_limit:.2f}")

print(f"\n→ See the ledger: https://figuard-sandbox-1.onrender.com/ui")